In [3]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, MultiLineString

# Load Klagenfurt network
network = gpd.read_file("klagenfurt_network.gpkg")

nodes_list = []
edges_list = []
node_id = 1
node_lookup = {}

def process_line(line):
    global node_id
    coords = list(line.coords)
    start = coords[0]
    end = coords[-1]
    for coord in (start, end):
        if coord not in node_lookup:
            node_lookup[coord] = node_id
            nodes_list.append({
                'node_id': node_id,
                'lat': coord[1],
                'lon': coord[0]
            })
            node_id += 1
    return node_lookup[start], node_lookup[end], line.length

for idx, row in network.iterrows():
    geom = row.geometry
    # Handle MultiLineString by iterating parts
    if isinstance(geom, MultiLineString):
        parts = geom.geoms
    else:
        parts = [geom]

    for part in parts:
        start_node, end_node, length = process_line(part)
        distance = length * 111000  # approximate meters
        travel_time = max(30, distance / (50 * 1000 / 3600))
        edges_list.append({
            'start_node': start_node,
            'end_node': end_node,
            'travel_time': travel_time,
            'distance': distance
        })

# Save nodes.csv and edges.csv
nodes_df = pd.DataFrame(nodes_list)
edges_df = pd.DataFrame(edges_list)

nodes_df.to_csv('nodes.csv', index=False)
edges_df.to_csv('edges.csv', index=False)

print(f"Created {len(nodes_df)} nodes and {len(edges_df)} edges")


Created 6880 nodes and 8900 edges
